# Gatekeeper, giai đoạn 0 (§37): Kaggle có chấm được task SWE-smith với cùng điểm không?

**Trước khi chạy** (khung Settings bên phải):
- **Internet: On** (cần xác minh số điện thoại);
- **Accelerator: None** (không cần GPU);
- Persistence: không cần.

Chạy lần lượt từng ô. Ô 4 mất khoảng 30–60 phút và **chạy tiếp được** nếu bị ngắt (chạy lại ô đó).
Không ô nào in khóa hay thông tin tài khoản. Kết quả nằm ở `/kaggle/working/`: tải ba file `k0*.json*` về và gửi lại.

In [ ]:
# Ô 1 — cài đặt (khoảng 2–3 phút)
!pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff 2>&1 | tail -2
!udocker --allow-root install 2>&1 | tail -1
!docker info > /dev/null 2>&1 && echo "Docker daemon: CO" || echo "Docker daemon: KHONG"
!df -h /kaggle/working | tail -1

In [ ]:
# Ô 2 — mã và dữ liệu (repo công khai; không cần đăng nhập)
import os
import subprocess

os.chdir("/kaggle/working")
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
if not os.path.isdir("gatekeeper"):
    url = f"https://github.com/{REPO}"
    branch = "claude/gatekeeper-13-research-nxytoh"  # Kaggle work lives here until it runs
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url], check=True)
RAW = f"https://raw.githubusercontent.com/{REPO}"
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
os.makedirs("k0data", exist_ok=True)
for name, path in files.items():
    dest = f"k0data/{name}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", "-o", dest, f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(dest), "byte")
head = subprocess.run(
    ["git", "-C", "gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print(head.stdout)

In [ ]:
# Ô 3 — K0a: image của task có chạy được không (khoảng 2–3 phút cho lần đầu)
import json
import subprocess
import sys

sys.path.insert(0, "/kaggle/working/gatekeeper/kaggle/scripts")
import udocker_shim  # noqa: E402

img = "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6"
name = udocker_shim.base_container(img)
inner = "git log --oneline | head -2; python --version; "
inner += "python -m pytest -q tests/test_api.py 2>&1 | tail -2"
cmd = ["udocker", "--allow-root", "--quiet", "run", "--nobanner", "--workdir=/testbed", name]
r = subprocess.run([*cmd, "/bin/bash", "-lc", inner], capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-300:])
with open("/kaggle/working/k0a.json", "w") as fh:
    json.dump({"returncode": r.returncode, "tail": r.stdout[-600:]}, fh)

In [ ]:
# Ô 4 — K0b: chấm lại các bản vá đã lưu, so với điểm đã ghi dưới Docker
# (30–60 phút; chạy lại được)
import os

os.makedirs("/kaggle/working/k0run", exist_ok=True)
os.chdir("/kaggle/working/k0run")
!python /kaggle/working/gatekeeper/kaggle/scripts/k0_stage0.py \
    --rows /kaggle/working/k0data/swesmith_pallets__jinja.jsonl \
    --results /kaggle/working/k0data/f1_probe.json /kaggle/working/k0data/f1_skill_probe.json \
    --controls 20 --out /kaggle/working/k0b_kaggle.jsonl 2>&1 | tail -40
!rm -rf /kaggle/working/k0run/logs

In [ ]:
# Ô 5 — K0c, K0d: hạn mức gọi model và giao diện (tốn vài xu hạn mức; KHÔNG in khóa)
import json
import os
import traceback

out = {
    "env_present": sorted(k for k in os.environ if k.startswith(("MODEL_PROXY", "LLM"))),
    "LLMS_AVAILABLE": os.environ.get("LLMS_AVAILABLE"),
    "LLM_DEFAULT": os.environ.get("LLM_DEFAULT"),
}
try:
    import kaggle_benchmarks as kbench

    out["version"] = getattr(kbench, "__version__", "?")
    out["api"] = sorted(n for n in dir(kbench) if not n.startswith("_"))
    try:
        llms = kbench.llms
        out["models"] = sorted(llms.keys()) if hasattr(llms, "keys") else str(llms)[:2000]
    except Exception as e:
        out["models_error"] = repr(e)[:300]
    llm = kbench.llm
    out["llm_default"] = str(getattr(llm, "name", llm))[:200]
    out["llm_api"] = sorted(n for n in dir(llm) if not n.startswith("_"))
    out["reply_1"] = str(llm.prompt("Reply with exactly the word: alpha"))[:200]
    # nhiều lượt: lượt 2 hỏi lại nội dung lượt 1 (có giữ lịch sử không?)
    q2 = "Which single word did you reply with in your previous message?"
    out["reply_2"] = str(llm.prompt(q2))[:200]
    for attr in ("usage", "last_usage", "cost", "last_response"):
        if hasattr(llm, attr):
            out[attr] = str(getattr(llm, attr))[:500]
except Exception:
    out["error"] = traceback.format_exc()[-1500:]
with open("/kaggle/working/k0cd.json", "w") as fh:
    json.dump(out, fh, indent=1, default=str)
print(json.dumps(out, indent=1, default=str)[:4000])

## Việc cần làm bằng tay (K0c, K0e)

1. Mở **Settings → mục Quotas** của tài khoản: ghi lại hạn mức AI theo ngày và theo tháng (ví dụ "10 USD / 100 USD"), và hạn mức GPU.
2. Sau khi chạy ô 5, xem hạn mức AI đã bị trừ bao nhiêu.
3. Tìm điều khoản dùng hạn mức AI (trang Benchmarks hoặc Terms): chép nguyên văn câu nói hạn mức được dùng cho việc gì.

Gửi lại: ba file `k0a.json`, `k0b_kaggle.jsonl`, `k0cd.json` trong `/kaggle/working/` (nút Download ở khung Output), cùng ba ghi chú trên.